In [2]:
import torch

import triton
import triton.language as tl
import os
import json

DEVICE = triton.runtime.driver.active.get_active_torch_device()

def get_config(config_file_path):
    if os.path.exists(config_file_path):
        with open(config_file_path) as f:
            return {int(key): val for key, val in json.load(f).items()}

@triton.jit
def matmul_kernel_3d(
        # Pointers to matrices
        a_ptr, b_ptr, c_ptr,
        # Matrix dimensions
        M, N, K,
        expert_ids_ptr,
        # The stride variables represent how much to increase the ptr by when moving by 1
        # element in a particular dimension. E.g. `stride_am` is how much to increase `a_ptr`
        # by to get the element one row down (A has M rows).
        stride_am, stride_ak,  #
        stride_be, stride_bk, stride_bn,  #
        stride_cm, stride_cn,
        top_k: tl.constexpr,  #
        # Meta-parameters
        BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, BLOCK_SIZE_K: tl.constexpr,  #
        GROUP_SIZE_M: tl.constexpr,  #
        # ACTIVATION: tl.constexpr  #
):
    """Kernel for computing the matmul C = A x B.
    A has shape (M, K), B has shape (K, N) and C has shape (M, N)
    """
    # -----------------------------------------------------------
    # Map program ids `pid` to the block of C it should compute.
    # This is done in a grouped ordering to promote L2 data reuse.
    # See above `L2 Cache Optimizations` section for details.
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    # ----------------------------------------------------------
    # Create pointers for the first blocks of A and B.
    # We will advance this pointer as we move in the K direction
    # and accumulate
    # `a_ptrs` is a block of [BLOCK_SIZE_M, BLOCK_SIZE_K] pointers
    # `b_ptrs` is a block of [BLOCK_SIZE_K, BLOCK_SIZE_N] pointers
    # See above `Pointer Arithmetic` section for details
    offs_am = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    offs_k = tl.arange(0, BLOCK_SIZE_K)
    off_experts = tl.load(expert_ids_ptr + pid_m).to(tl.int64)
    a_ptrs = a_ptr + (offs_am[:, None] // top_k * stride_am + offs_k[None, :] * stride_ak)
    # b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)
    b_ptrs = b_ptr + off_experts * stride_be + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)

    # -----------------------------------------------------------
    # Iterate to compute a block of the C matrix.
    # We accumulate into a `[BLOCK_SIZE_M, BLOCK_SIZE_N]` block
    # of fp32 values for higher accuracy.
    # `accumulator` will be converted back to fp16 after the loop.
    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        # Load the next block of A and B, generate a mask by checking the K dimension.
        # If it is out of bounds, set it to 0.
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_SIZE_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_SIZE_K, other=0.0)
        # We accumulate along the K dimension.
        accumulator = tl.dot(a, b, accumulator)
        # Advance the ptrs to the next K block.
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk
    # You can fuse arbitrary activation functions here
    # while the accumulator is still in FP32!
    # if ACTIVATION == "leaky_relu":
    #     accumulator = leaky_relu(accumulator)
    c = accumulator.to(tl.float16)

    # -----------------------------------------------------------
    # Write back the block of the output matrix C with masks.
    offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
    c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
    tl.store(c_ptrs, c, mask=c_mask)

@triton.jit
def matmul_kernel_2d(
        # Pointers to matrices
        a_ptr, b_ptr, c_ptr,
        # Matrix dimensions
        M, N, K,
        # The stride variables represent how much to increase the ptr by when moving by 1
        # element in a particular dimension. E.g. `stride_am` is how much to increase `a_ptr`
        # by to get the element one row down (A has M rows).
        stride_am, stride_ak,  #
        stride_bk, stride_bn,  #
        stride_cm, stride_cn,
        # Meta-parameters
        BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, BLOCK_SIZE_K: tl.constexpr,  #
        GROUP_SIZE_M: tl.constexpr,  #
):
    """Kernel for computing the matmul C = A x B.
    A has shape (M, K), B has shape (K, N) and C has shape (M, N)
    """
    # -----------------------------------------------------------
    # Map program ids `pid` to the block of C it should compute.
    # This is done in a grouped ordering to promote L2 data reuse.
    # See above `L2 Cache Optimizations` section for details.
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    # -----------------------------------------------------------
    # Add some integer bound assumptions.
    # This helps to guide integer analysis in the backend to optimize
    # load/store offset address calculation
    tl.assume(pid_m >= 0)
    tl.assume(pid_n >= 0)
    tl.assume(stride_am > 0)
    tl.assume(stride_ak > 0)
    tl.assume(stride_bn > 0)
    tl.assume(stride_bk > 0)
    tl.assume(stride_cm > 0)
    tl.assume(stride_cn > 0)

    # ----------------------------------------------------------
    # Create pointers for the first blocks of A and B.
    # We will advance this pointer as we move in the K direction
    # and accumulate
    # `a_ptrs` is a block of [BLOCK_SIZE_M, BLOCK_SIZE_K] pointers
    # `b_ptrs` is a block of [BLOCK_SIZE_K, BLOCK_SIZE_N] pointers
    # See above `Pointer Arithmetic` section for details
    offs_am = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    offs_k = tl.arange(0, BLOCK_SIZE_K)
    a_ptrs = a_ptr + (offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)

    # -----------------------------------------------------------
    # Iterate to compute a block of the C matrix.
    # We accumulate into a `[BLOCK_SIZE_M, BLOCK_SIZE_N]` block
    # of fp32 values for higher accuracy.
    # `accumulator` will be converted back to fp16 after the loop.
    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        # Load the next block of A and B, generate a mask by checking the K dimension.
        # If it is out of bounds, set it to 0.
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_SIZE_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_SIZE_K, other=0.0)
        # We accumulate along the K dimension.
        accumulator = tl.dot(a, b, accumulator)
        # Advance the ptrs to the next K block.
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk

    c = accumulator.to(tl.float16)

    # -----------------------------------------------------------
    # Write back the block of the output matrix C with masks.
    offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
    c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
    tl.store(c_ptrs, c, mask=c_mask)

def matmul(a, b, num_experts, config):
    if num_experts > 1:
        top_k_num = 1
        # Check constraints.
        assert a.shape[1] == b.shape[2], "Incompatible dimensions"
        assert a.is_contiguous(), "Matrix A must be contiguous"
        assert b.shape[0] == num_experts, "Number of experts does not match"
        M, K = a.shape
        num_experts, N, K = b.shape
        # Allocates output.
        c = torch.empty((M, top_k_num, N), device=a.device, dtype=torch.float16)
        # 1D launch kernel where each block gets its own program.
        grid = lambda META: (triton.cdiv(M, META['BLOCK_SIZE_M']) * triton.cdiv(N, META['BLOCK_SIZE_N']), )
        activated_experts = min(M, num_experts)
        expert_ids = torch.arange(activated_experts, device=a.device, dtype=torch.int32).view(1, -1)

        matmul_kernel_3d[grid](
            a, b, c,  #
            M, N, K,  #
            expert_ids, 
            a.stride(0), a.stride(1),  #
            b.stride(0), b.stride(2), b.stride(1),  #
            c.stride(1), c.stride(2),  #
            top_k=top_k_num,  #
            # ACTIVATION=activation  #
            **config
        )
    else:
        # Check constraints.
        assert a.shape[1] == b.shape[0], "Incompatible dimensions"
        assert a.is_contiguous(), "Matrix A must be contiguous"
        M, K = a.shape
        K, N = b.shape
        # Allocates output.
        c = torch.empty((M, N), device=a.device, dtype=torch.float16)
        # 1D launch kernel where each block gets its own program.
        grid = lambda META: (triton.cdiv(M, META['BLOCK_SIZE_M']) * triton.cdiv(N, META['BLOCK_SIZE_N']), )
        matmul_kernel_2d[grid](
            a, b, c,  #
            M, N, K,  #
            a.stride(0), a.stride(1),  #
            b.stride(0), b.stride(1),  #
            c.stride(0), c.stride(1),  #
            **config
        )
    return c

@triton.jit
def grouped_matmul_kernel(
    # device tensor of matrices pointers
    group_a_ptrs,
    group_b_ptrs,
    group_c_ptrs,
    # device tensor of gemm sizes. its shape is [group_size, 3]
    # dim 0 is group_size, dim 1 is the values of <M, N, K> of each gemm
    group_gemm_sizes,
    # device tensor of leading dimension sizes. its shape is [group_size, 3]
    # dim 0 is group_size, dim 1 is the values of <lda, ldb, ldc> of each gemm
    g_lds,
    # number of gemms
    group_size,
    # number of virtual SM
    NUM_SM: tl.constexpr,
    # tile sizes
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr,
):
    tile_idx = tl.program_id(0)
    last_problem_end = 0
    for g in range(group_size):
        # get the gemm size of the current problem
        gm = tl.load(group_gemm_sizes + g * 3)
        gn = tl.load(group_gemm_sizes + g * 3 + 1)
        gk = tl.load(group_gemm_sizes + g * 3 + 2)
        num_m_tiles = tl.cdiv(gm, BLOCK_SIZE_M)
        num_n_tiles = tl.cdiv(gn, BLOCK_SIZE_N)
        num_tiles = num_m_tiles * num_n_tiles
        # iterate through the tiles in the current gemm problem
        while (tile_idx >= last_problem_end and tile_idx < last_problem_end + num_tiles):
            # pick up a tile from the current gemm problem
            k = gk
            lda = tl.load(g_lds + g * 3)
            ldb = tl.load(g_lds + g * 3 + 1)
            ldc = tl.load(g_lds + g * 3 + 2)
            a_ptr = tl.load(group_a_ptrs + g).to(tl.pointer_type(tl.float16))
            b_ptr = tl.load(group_b_ptrs + g).to(tl.pointer_type(tl.float16))
            c_ptr = tl.load(group_c_ptrs + g).to(tl.pointer_type(tl.float16))
            # figure out tile coordinates
            tile_idx_in_gemm = tile_idx - last_problem_end
            tile_m_idx = tile_idx_in_gemm // num_n_tiles
            tile_n_idx = tile_idx_in_gemm % num_n_tiles

            # do regular gemm here
            offs_am = tile_m_idx * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
            offs_bn = tile_n_idx * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
            offs_k = tl.arange(0, BLOCK_SIZE_K)
            a_ptrs = a_ptr + offs_am[:, None] * lda + offs_k[None, :]
            b_ptrs = b_ptr + offs_k[:, None] * ldb + offs_bn[None, :]
            accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
            for kk in range(0, tl.cdiv(k, BLOCK_SIZE_K)):
                # hint to Triton compiler to do proper loop pipelining
                tl.multiple_of(a_ptrs, [16, 16])
                tl.multiple_of(b_ptrs, [16, 16])
                # assume full tile for now
                a = tl.load(a_ptrs)
                b = tl.load(b_ptrs)
                accumulator += tl.dot(a, b)
                a_ptrs += BLOCK_SIZE_K
                b_ptrs += BLOCK_SIZE_K * ldb
            c = accumulator.to(tl.float16)

            offs_cm = tile_m_idx * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
            offs_cn = tile_n_idx * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
            c_ptrs = c_ptr + ldc * offs_cm[:, None] + offs_cn[None, :]

            # assumes full tile for now
            tl.store(c_ptrs, c)

            # go to the next tile by advancing NUM_SM
            tile_idx += NUM_SM

        # get ready to go to the next gemm problem
        last_problem_end = last_problem_end + num_tiles

if __name__ == "__main__":
    K = 5120
    N = 2048 # divde by 8 and multipyl by 2
    niter= 12
    use_fp_8 = True
    configs = get_config(config_file_path = "/home/ubuntu/vllm/benchmarks/kernels/config.json")
    for M in [1,2,8,16,32,64, 128, 256, 512]:
        num_experts = min(M, 128)
        config = configs[M]
        a = torch.randn((M, K), device=DEVICE, dtype=torch.float16)
        if num_experts > 1:
            b = torch.randn((num_experts, N, K), device=DEVICE, dtype=torch.float16)
        else:
            b = torch.randn((K, N), device=DEVICE, dtype=torch.float16)

        if use_fp_8:
            a = a.to(torch.float8_e4m3fn)
            # b = b.T
            b = b.to(torch.float8_e4m3fn)
        quantiles = [0.5, 0.2, 0.8]

        if not use_fp_8:
            cublas_ms = triton.testing.do_bench(lambda: torch.matmul(a, b), quantiles=quantiles)
            triton_ms = triton.testing.do_bench(lambda: matmul(a, b, num_experts, config), quantiles=quantiles)
            print("M", M, "cublasms", cublas_ms)
        else:
            triton_ms = triton.testing.do_bench(lambda: matmul(a, b, num_experts, config), quantiles=quantiles)
        print("M", M, "tritonms", triton_ms)

M 1 tritonms [0.018223999999463558, 0.017983999103307724, 0.018598400056362152]
M 2 tritonms [0.0226879995316267, 0.02236800082027912, 0.023072000592947006]
M 8 tritonms [0.02051199972629547, 0.020128000527620316, 0.021536000072956085]
M 16 tritonms [0.022816000506281853, 0.022624000906944275, 0.023104000836610794]
M 32 tritonms [0.025760000571608543, 0.025536000728607178, 0.02630399912595749]
M 64 tritonms [0.028704000636935234, 0.028416000306606293, 0.029311999678611755]
M 128 tritonms [0.04572800174355507, 0.045343998819589615, 0.04646399989724159]
M 256 tritonms [0.07459200173616409, 0.07388799637556076, 0.07503999769687653]
M 512 tritonms [0.12889599800109863, 0.1284479945898056, 0.12942719757556914]


In [4]:
matmul(a, b, num_experts, config)

tensor([[[ 141.7500,  -64.1875,   12.2578,  ...,  -13.6953, -112.5000,
            42.3750]],

        [[  17.6719,    4.6875,  -50.7812,  ...,    9.0703,   56.5625,
             2.7031]],

        [[  -9.1250,  160.5000,   12.0547,  ...,    9.7656,  -10.6641,
           -81.4375]],

        ...,

        [[   7.9727,  -17.0938,   51.5938,  ...,   44.9375,   -4.8945,
            56.0938]],

        [[ 109.0625,  -26.3750,  -37.5938,  ...,   -8.8125,   75.8750,
            39.6250]],

        [[ -65.5625,   41.0625,   91.1250,  ...,   55.0000,  -51.2812,
            41.0625]]], device='cuda:0', dtype=torch.float16)

In [5]:
A = torch.rand((M, K), device=DEVICE, dtype=torch.float16)
B = torch.rand((K, N), device=DEVICE, dtype=torch.float16)
A.stride(0), B.stride(0)

(5120, 2048)

In [6]:
# only launch the kernel, no tensor preparation here to remove all overhead
def triton_perf_fn(a_ptrs, b_ptrs, c_ptrs, sizes, lds, group_size, config):
    grid = lambda META: (META['NUM_SM'],)
    grouped_matmul_kernel[grid](
        a_ptrs,
        b_ptrs,
        c_ptrs,
        sizes,
        lds,
        group_size,
        **config,
    )

In [7]:
def group_gemm_fn(group_A, group_B, use_config=False):
    assert len(group_A) == len(group_B)
    group_size = len(group_A)

    A_addrs = []
    B_addrs = []
    C_addrs = []
    g_sizes = []
    g_lds = []
    group_C = []
    for i in range(group_size):
        A = group_A[i]
        B = group_B[i]
        assert A.shape[1] == B.shape[0]
        M, K = A.shape
        K, N = B.shape
        C = torch.empty((M, N), device=DEVICE, dtype=A.dtype)
        group_C.append(C)
        A_addrs.append(A.data_ptr())
        B_addrs.append(B.data_ptr())
        C_addrs.append(C.data_ptr())
        g_sizes += [M, N, K]
        g_lds += [A.stride(0), B.stride(0), C.stride(0)]

    # print(A_addrs, B_addrs, C_addrs)
    # note these are device tensors
    d_a_ptrs = torch.tensor(A_addrs, device=DEVICE)
    d_b_ptrs = torch.tensor(B_addrs, device=DEVICE)
    d_c_ptrs = torch.tensor(C_addrs, device=DEVICE)
    d_g_sizes = torch.tensor(g_sizes, dtype=torch.int32, device=DEVICE)
    d_g_lds = torch.tensor(g_lds, dtype=torch.int32, device=DEVICE)
    # we use a fixed number of CTA, and it's auto-tunable
    config = configs[M]
    config['NUM_SM']= num_sms() 
    config.pop('GROUP_SIZE_M', None)
    # print(config)
    grid = lambda META: (META['NUM_SM'], )
    if use_config:
        grouped_matmul_kernel[grid](
        d_a_ptrs,
        d_b_ptrs,
        d_c_ptrs,
        (M, N, K),
        (K, N, N),
        group_size,
        **config,
        )
    else:
        grouped_matmul_kernel[grid](
        d_a_ptrs,
        d_b_ptrs,
        d_c_ptrs,
        d_g_sizes,
        d_g_lds,
        group_size
    )
    return group_C

In [5]:

def num_sms():
    if is_cuda():
        return torch.cuda.get_device_properties("cuda").multi_processor_count
    return 148

def is_cuda():
    return triton.runtime.driver.active.get_current_target().backend == "cuda"

# run MoE test
def test_moe(M=1, N=2048, K=5192, num_experts=128):
    group_a = []
    group_b = []
    num_activated_experts = min(num_experts, M)
    #print(M, num_activated_experts)
    a = torch.rand((M, K), device=DEVICE, dtype=torch.float16)
    b = torch.rand((num_experts, K, N), device=DEVICE, dtype=torch.float16)
    expert_ids = torch.arange(num_activated_experts, device=a.device, dtype=torch.int32) #.view(-1,1)
    for m in range(num_activated_experts):
        group_a.append(torch.unsqueeze(a[m,:], 0))
        #print(b[expert_ids[m], :, :].shape, torch.unsqueeze(a[m,:], 0).shape)
        group_b.append(b[expert_ids[m], :, :])
        # print(group_a[m].shape, group_b[m].shape)

    quantiles = [0.5, 0.2, 0.8]
    triton_ms = triton.testing.do_bench(lambda: group_gemm_fn(group_a, group_b, use_config=True), quantiles=quantiles)
    #ref_out = [torch.matmul(a, b) for a, b in zip(group_a, group_b)]
    # for i in range(num_activated_experts):
    #     assert torch.allclose(ref_out[i], tri_out[i], atol=1e-2, rtol=1e-2)
    print(triton_ms)

In [84]:
@triton.jit
def mat_vec_kernel(
    vec_ptr,
    matrix_ptr,
    out_ptr,
    vec_stridex,
    matrix_stridey,
    matrix_stridex,
    out_stridex,
    NUM_SM: tl.constexpr,
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr,
):
    vec_ptr = vec_ptr + vec_stridex * tl.arange(0, BLOCK_SIZE_M)
    #vec_ptr = tl.reshape(vec_ptr, (BLOCK_SIZE_M, 1))

    out_ptr = out_ptr + out_stridex * tl.arange(0, BLOCK_SIZE_M)
    #out_ptr = tl.reshape(out_ptr, (BLOCK_SIZE_M, 1))

    matrix_x = matrix_stridex * tl.arange(0, BLOCK_SIZE_M)
    matrix_y = matrix_stridey * tl.arange(0, BLOCK_SIZE_M)

    matrix_ptr = matrix_ptr + (matrix_x[None, :] + matrix_y[:, None])
    # TODO: add in 
    # tl.sum(val[:, None] * matrix, 0)

    val = tl.load(vec_ptr).to(tl.float32)
    matrix = tl.load(matrix_ptr).to(tl.float32)
    tl.store(out_ptr, tl.sum(val[:, None] * matrix, 0))

In [18]:
@triton.jit
def matmul_kernel(
        # Pointers to matrices
        a_ptr, b_ptr, c_ptr,
        # Matrix dimensions
        M, N, K,
        # The stride variables represent how much to increase the ptr by when moving by 1
        # element in a particular dimension. E.g. `stride_am` is how much to increase `a_ptr`
        # by to get the element one row down (A has M rows).
        stride_am, stride_ak,  #
        stride_bk, stride_bn,  #
        stride_cm, stride_cn,
        # Meta-parameters
        BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, BLOCK_SIZE_K: tl.constexpr,  #
        GROUP_SIZE_M: tl.constexpr,  #
        ACTIVATION: tl.constexpr  #
):
    """Kernel for computing the matmul C = A x B.
    A has shape (M, K), B has shape (K, N) and C has shape (M, N)
    """
    # -----------------------------------------------------------
    # Map program ids `pid` to the block of C it should compute.
    # This is done in a grouped ordering to promote L2 data reuse.
    # See above `L2 Cache Optimizations` section for details.
    # Program ID
    pid = tl.program_id(axis=0)
    # Number of program ids along the M axis
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    # Number of programs ids along the N axis
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    # Number of programs in group
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    # Id of the group this program is in
    group_id = pid // num_pid_in_group
    # Row-id of the first program in the group
    first_pid_m = group_id * GROUP_SIZE_M
    # If `num_pid_m` isn't divisible by `GROUP_SIZE_M`, the last group is smaller
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    # *Within groups*, programs are ordered in a column-major order
    # Row-id of the program in the *launch grid*
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    # Col-id of the program in the *launch grid*
    pid_n = (pid % num_pid_in_group) // group_size_m

    # -----------------------------------------------------------
    # Add some integer bound assumptions.
    # This helps to guide integer analysis in the backend to optimize
    # load/store offset address calculation
    tl.assume(pid_m >= 0)
    tl.assume(pid_n >= 0)
    tl.assume(stride_am > 0)
    tl.assume(stride_ak > 0)
    tl.assume(stride_bn > 0)
    tl.assume(stride_bk > 0)
    tl.assume(stride_cm > 0)
    tl.assume(stride_cn > 0)

    # ----------------------------------------------------------
    # Create pointers for the first blocks of A and B.
    # We will advance this pointer as we move in the K direction
    # and accumulate
    # `a_ptrs` is a block of [BLOCK_SIZE_M, BLOCK_SIZE_K] pointers
    # `b_ptrs` is a block of [BLOCK_SIZE_K, BLOCK_SIZE_N] pointers
    # See above `Pointer Arithmetic` section for details
    offs_am = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    offs_k = tl.arange(0, BLOCK_SIZE_K)
    a_ptrs = a_ptr + (offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)

    # -----------------------------------------------------------
    # Iterate to compute a block of the C matrix.
    # We accumulate into a `[BLOCK_SIZE_M, BLOCK_SIZE_N]` block
    # of fp32 values for higher accuracy.
    # `accumulator` will be converted back to fp16 after the loop.
    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        # Load the next block of A and B, generate a mask by checking the K dimension.
        # If it is out of bounds, set it to 0.
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_SIZE_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_SIZE_K, other=0.0)
        # We accumulate along the K dimension.
        accumulator = tl.dot(a, b, accumulator)
        # Advance the ptrs to the next K block.
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk
    # You can fuse arbitrary activation functions here
    # while the accumulator is still in FP32!
    if ACTIVATION == "leaky_relu":
        accumulator = leaky_relu(accumulator)
    c = accumulator.to(tl.float16)

    # -----------------------------------------------------------
    # Write back the block of the output matrix C with masks.
    offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
    c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
    tl.store(c_ptrs, c, mask=c_mask)

In [31]:
def matmul(a, b, activation=""):
    # Check constraints.
    assert a.shape[1] == b.shape[0], "Incompatible dimensions"
    assert a.is_contiguous(), "Matrix A must be contiguous"
    M, K = a.shape
    K, N = b.shape
    configs = get_config(config_file_path = "/home/ubuntu/vllm/benchmarks/kernels/config.json")
    config = configs[M]
    # Allocates output.
    c = torch.empty((M, N), device=a.device, dtype=torch.float16)
    # 1D launch kernel where each block gets its own program.
    grid = lambda META: (triton.cdiv(M, META['BLOCK_SIZE_M']) * triton.cdiv(N, META['BLOCK_SIZE_N']), )
    matmul_kernel[grid](
        a, b, c,  #
        M, N, K,  #
        a.stride(0), a.stride(1),  #
        b.stride(0), b.stride(1),  #
        c.stride(0), c.stride(1),  #
        ACTIVATION=activation,  #
        **config
    )
    return c

In [ ]:
def test_moe_mat_vec(M=1, N=2048, K=5192, num_experts=128, use_fp8=False,
                     dtype_fp8 = torch.float8_e4m3fn, is_mat_vec=False):
    group_A = []
    group_B = []
    A_addrs = []
    B_addrs = []
    C_addrs = []
    g_sizes = []
    g_lds = []
    group_C = []
    num_activated_experts = min(num_experts, M)
    A_total = torch.rand((M, K), device=DEVICE, dtype=torch.float16)
    B_total = torch.rand((num_experts, K, N), device=DEVICE, dtype=torch.float16)
    expert_ids = torch.arange(num_activated_experts, device=a.device, dtype=torch.int32) #.view(-1,1)
    times = torch.zeros((num_activated_experts, 3))
    for m in range(num_activated_experts):
        A = torch.unsqueeze(A_total[m,:], 0)
        M_e = A.shape[0]
        B = B_total[expert_ids[m], :, :]
        if use_fp8:
            A = A.to(dtype_fp8)
            # b = b.T
            B = B.to(dtype_fp8)
        C = torch.empty((M_e, N), device=DEVICE, dtype=torch.float16)

       # B_T = B.T.contiguous()
        group_A.append(A)
        group_B.append(B)
        # group_B_T.append(B_T)
        group_C.append(C)

        quantiles = [0.5, 0.2, 0.8]
        if is_mat_vec:
            configs = get_config(config_file_path = "/home/ubuntu/vllm/benchmarks/kernels/config.json")
            config = configs[M_e]
            config['NUM_SM']= num_sms() 
            config.pop('GROUP_SIZE_M', None)
            config.pop('num_warps', None)
            config.pop('num_stages', None)
            grid = (1,)
            times[m,0], times[m,1], times[m,2] = triton.testing.do_bench(
            lambda: mat_vec_kernel[grid](A, B, C, A.stride(1), B.stride(0), B.stride(1), C.stride(1),**config), quantiles=quantiles)
        else:
            configs = get_config(config_file_path = "/home/ubuntu/vllm/benchmarks/kernels/config.json")
            config=configs[M_e]
            # times[m,0], times[m,1], times[m,2] = triton.testing.do_bench(
            # lambda: matmul(A,B), quantiles=quantiles)
            grid = lambda META: (triton.cdiv(M, META['BLOCK_SIZE_M']) * triton.cdiv(N, META['BLOCK_SIZE_N']), )
            times[m,0], times[m,1], times[m,2] = triton.testing.do_bench(lambda: matmul_kernel[grid](
            A, B, C,  #
            M_e, N, K,  #
            A.stride(0), A.stride(1),  #
            B.stride(0), B.stride(1),  #
            C.stride(0), C.stride(1),  #
            ACTIVATION="",
            **config) ,
            quantiles=quantiles)
    # #ref_out = [torch.matmul(a, b) for a, b in zip(group_a, group_b)]
    # # for i in range(num_activated_experts):
    # #     assert torch.allclose(ref_out[i], tri_out[i], atol=1e-2, rtol=1e-2)
    return torch.sum(times, axis=0) #, group_A, group_B, group_C

In [80]:
M = 2

In [86]:
t, A,B,C = test_moe_mat_vec(M)

1
1


In [87]:
C

[tensor([[   6.2617,    4.6367,    4.1250,  ..., 1267.0000, 1296.0000,
          1297.0000]], device='cuda:0', dtype=torch.float16),
 tensor([[   2.0273,    3.2422,    2.8203,  ..., 1274.0000, 1280.0000,
          1285.0000]], device='cuda:0', dtype=torch.float16)]

In [88]:
torch.matmul(A[1],B[1])

tensor([[1297., 1296., 1278.,  ..., 1260., 1281., 1269.]], device='cuda:0',
       dtype=torch.float16)

In [95]:
niter = 10
for i in range(niter):
    M = 2**i
    ms, max_ms, min_ms = test_moe_mat_vec(M)
    print("M", M, ms, max_ms, min_ms)

M 1 tensor(0.0431) tensor(0.0429) tensor(0.0436)
M 2 tensor(0.0723) tensor(0.0717) tensor(0.0731)
M 4 tensor(0.1451) tensor(0.1436) tensor(0.1476)
M 8 tensor(0.2959) tensor(0.2908) tensor(0.2999)
M 16 tensor(0.6057) tensor(0.5973) tensor(0.6155)
M 32 tensor(1.1622) tensor(1.1394) tensor(1.1887)
M 64 tensor(2.4326) tensor(2.3885) tensor(2.4866)
M 128 tensor(6.4025) tensor(6.3496) tensor(6.4819)
M 256 tensor(11.7996) tensor(11.7504) tensor(11.8513)
M 512 tensor(22.1286) tensor(22.0795) tensor(22.1817)


In [59]:
0.04*128

5.12

In [15]:
c

tensor([[1258., 1281., 1296.,  ..., 1296., 1278., 1313.]], device='cuda:0',
       dtype=torch.float16)

In [16]:
group_C

[tensor([[  3.8555,   3.9766,   4.9531,  ..., -60.2812,   2.0117, -38.9062]],
        device='cuda:0', dtype=torch.float16)]

In [6]:
def test_moe_perf(M=1, N=2048, K=5192, num_experts=128, use_fp8=False, dtype_fp8 = torch.float8_e4m3fn):
    group_A = []
    group_B = []
    group_B_T = []
    A_addrs = []
    B_addrs = []
    B_T_addrs = []
    C_addrs = []
    g_sizes = []
    g_lds = []
    g_T_lds = []
    group_C = []
    strides_A = []
    strides_B = []
    strides_C = []
    num_activated_experts = min(num_experts, M)
    print(M, num_activated_experts)
    A_total = torch.rand((M, K), device=DEVICE, dtype=torch.float16)
    B_total = torch.rand((num_experts, K, N), device=DEVICE, dtype=torch.float16)
    config = configs[M]
    config['NUM_SM']= num_sms() 
    config.pop('GROUP_SIZE_M', None)
    expert_ids = torch.arange(num_activated_experts, device=a.device, dtype=torch.int32) #.view(-1,1)
    for m in range(num_activated_experts):
        A = torch.unsqueeze(A_total[m,:], 0)
        print(A.shape)
        M_e = A.shape[0]
        print(M_e)
        B = B_total[expert_ids[m], :, :]
        if use_fp8:
            A = A.to(dtype_fp8)
            # b = b.T
            B = B.to(dtype_fp8)
        C = torch.empty((M_e, N), device=DEVICE, dtype=torch.float16)
        print(C.shape)

       # B_T = B.T.contiguous()
        group_A.append(A)
        group_B.append(B)
        # group_B_T.append(B_T)
        group_C.append(C)
        A_addrs.append(A.data_ptr())
        B_addrs.append(B.data_ptr())
        # B_T_addrs.append(B_T.data_ptr())
        C_addrs.append(C.data_ptr())
        g_sizes += [M_e, N, K]
        g_lds += [A.stride(0), B.stride(0), C.stride(0)]
       # g_T_lds += [A.stride(0), B_T.stride(0), C.stride(0)]

    d_a_ptrs = torch.tensor(A_addrs, device=DEVICE)
    d_b_ptrs = torch.tensor(B_addrs, device=DEVICE)
   # d_b_t_ptrs = torch.tensor(B_T_addrs, device=DEVICE)
    d_c_ptrs = torch.tensor(C_addrs, device=DEVICE)
    d_g_sizes = torch.tensor(g_sizes, dtype=torch.int32, device=DEVICE)
    d_g_lds = torch.tensor(g_lds, dtype=torch.int32, device=DEVICE)
    # d_g_t_lds = torch.tensor(g_T_lds, dtype=torch.int32, device=DEVICE)

    quantiles = [0.5, 0.2, 0.8]
    ms, min_ms, max_ms = triton.testing.do_bench(
            lambda: triton_perf_fn(d_a_ptrs, d_b_ptrs, d_c_ptrs, d_g_sizes, d_g_lds, num_activated_experts, config), quantiles=quantiles)
    # #ref_out = [torch.matmul(a, b) for a, b in zip(group_a, group_b)]
    # # for i in range(num_activated_experts):
    # #     assert torch.allclose(ref_out[i], tri_out[i], atol=1e-2, rtol=1e-2)
    print(ms, min_ms, max_ms)

In [7]:
test_moe_perf(M=1)

1 1
torch.Size([1, 5192])
1
torch.Size([1, 2048])


NameError: name 'triton_perf_fn' is not defined

In [84]:
group_C[0]

tensor([[17.2500, 16.0000, 19.2500,  ...,  0.8892,  0.7148,  0.1704]],
       device='cuda:0', dtype=torch.float16)

In [83]:
c

tensor([[1308., 1326., 1311.,  ..., 1331., 1329., 1320.]], device='cuda:0',
       dtype=torch.float16)

In [1]:
test_moe()

NameError: name 'test_moe' is not defined

In [10]:
test_moe_perf()

1 1
1


RuntimeError: CUDA error: an illegal memory access was encountered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
